In [1]:
from curator.simulate import MLCalculator
from curator.simulate.core.simulator import Simulator

from curator.simulate.callbacks.thermo_uncertainty import ThermoWithUncertainty
from curator.simulate.callbacks.thermo_uncertainty import ThermoWithUncertainty
from ase.md.langevin import Langevin
from ase.io import read, write

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/opt/conda/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


In [2]:
# it is possible to directly use a trained model to generate an ASE calculator
atoms = read('../example/LiFePO4.traj')
calc = MLCalculator('../example/train/model_path')

# After assigning calculator to the atoms, you can calculate many properties with ASE
atoms.calc = calc
atoms.get_potential_energy()

/workspace/curator/curator/utils.py:91: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  obj = torch.load(model_file, map_location=torch.device(device))


-179.77609252929688

You need to create a engine to run the desired simulation. The engine can be ase's dynamics or other engines. It is also possible to define a engine by yourself through inheriting the base `Engine` class

In [3]:
from curator.simulate.engines.ase_md import MDEngine
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

MaxwellBoltzmannDistribution(atoms, temperature_K=300)
# dyn = Langevin(atoms, timestep=0.5, friction=0.2, temperature_K=300)  # not recommended to create engine in this way
engine = MDEngine('ase.md.langevin.Langevin', timestep=0.5, friction=0.2, temperature_K=300)
# engine.setup(atoms)
# engine.run(100)

Although a simple `Engine` class is capable of some simple simulation tasks, it is inconvenient to do some pre-process and post-process for the atoms, calculator, and also logging some per-step simulation information.

That's why we created the `Simulator` class, which works in a way alike to [pytorch lightning](https://lightning.ai/docs/pytorch/stable/).

You can easily add some callbacks into a simulator. These callbacks is very flexible and powerful for different usages.

Here we will showcase some useful callbacks which are able to monitor a MD simulation, save structures when simulating, and pre-process the initial structures.

In [4]:
from curator.simulate.callbacks import MDThermoLogger
from curator.simulate.callbacks import CalculatorAssign

In [5]:
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,   # output log to screen
    format="%(asctime)s - %(levelname)s - %(message)s"
)

thermo_logger = MDThermoLogger()       # define a default logger
calc_cb = CalculatorAssign(calc)       # assign calculator
simulator = Simulator(
    init_traj='../example/LiFePO4.traj',
    engine=engine,
    callbacks=[thermo_logger, calc_cb],
)

In [9]:
simulator.run(100)

2025-11-20 16:11:06,354 - INFO -            step           epot           ekin           etot
2025-11-20 16:11:06,355 - INFO - Calcator assigned to atoms.
2025-11-20 16:11:06,376 - INFO -             302     -182.97382        0.98200     -181.99182
2025-11-20 16:11:06,394 - INFO -             303     -183.07137        1.08213     -181.98923
2025-11-20 16:11:06,413 - INFO -             304     -183.08203        0.98369     -182.09834
2025-11-20 16:11:06,431 - INFO -             305     -183.15547        0.99388     -182.16159
2025-11-20 16:11:06,448 - INFO -             306     -183.24681        0.95600     -182.29081
2025-11-20 16:11:06,466 - INFO -             307     -183.28333        0.95997     -182.32336
2025-11-20 16:11:06,483 - INFO -             308     -183.16905        0.91552     -182.25354
2025-11-20 16:11:06,500 - INFO -             309     -183.04674        0.85504     -182.19170
2025-11-20 16:11:06,517 - INFO -             310     -183.19286        1.06243     -182.13042

In most cases, we may care about the uncertainties of the structures generated in a simulation. We can add an uncertainty callback to achieve that.

In [19]:
atoms.get_potential_energy()

-182.91700744628906

In [20]:
MaxwellBoltzmannDistribution(atoms, temperature_K=300)
dyn = Langevin(atoms, timestep=0.5, friction=0.2, temperature_K=300)

In [27]:
dyn.run(100)

True

In [28]:
atoms.get_potential_energy()

-182.99713134765625

In [14]:
simulator.run(steps=100)

2025-11-20 15:03:15,722 - INFO -            step           epot           ekin           etot
2025-11-20 15:03:15,745 - INFO -               1     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,762 - INFO -               2     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,779 - INFO -               3     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,795 - INFO -               4     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,811 - INFO -               5     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,827 - INFO -               6     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,843 - INFO -               7     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,859 - INFO -               8     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,876 - INFO -               9     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,892 - INFO -              10     -183.65

In [ ]:
import 

In [ ]:
from typing import Any

In [ ]:
isinstance(_resolve('ase.md.langevin.Langevin'), type)